<!-- # Gold Layer - Analytics & Business Intelligence

## Arquitetura Medallion - Camada Gold

Este notebook implementa a camada **Gold** com modelos dimensionais e agregações para BI.

### Objetivos

1. **Modelos Dimensionais**: Star schema para análise de negócio
2. **Agregações**: Pre-computed metrics para performance
3. **KPIs**: Key Performance Indicators do negócio
4. **Views Materializadas**: Queries otimizadas para dashboards
5. **SCD Type 2**: Slowly Changing Dimensions com histórico

### Arquitetura

```

 BRONZE (default) 
 • 6 tables ERP 
 • 3.6M+ rows 

 
 ↓ run_silver_pipeline.py (pandas)

 SILVER (track_silver) 
 • 6 clean tables 
 • 50,027 rows 
 • Quality: 91.7% avg 
 • 3 metrics tables 

 
 ↓ populate_fact_vendas.py

 GOLD (track_gold) 
 • 2 dimensions (2,560 records) 
 • 1 fact (10,108 vendas) 
 • 2 aggregations (46 days) 
 • 3 KPIs 
 • 3 analytical views 

 
 ↓ incremental_etl.py
 Watermark-based incremental updates -->

---
## 1. Imports e Configuração

In [79]:
from dotenv import load_dotenv
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import clickhouse_connect
import numpy as np
import pandas as pd
import json
from typing import Dict, List
from datetime import datetime, timedelta
from pathlib import Path
import sys
import os
import warnings
warnings.filterwarnings('ignore')


load_dotenv()

print("Imports carregados!")

Imports carregados!


---
## 2. Conexão ClickHouse

In [80]:
CH_HOST = os.getenv('CLICKHOUSE_HOST',
                    'e1a1lieug8.us-central1.gcp.clickhouse.cloud')
CH_PORT = int(os.getenv('CLICKHOUSE_PORT', 8443))
CH_USER = os.getenv('CLICKHOUSE_USER', 'default')
CH_PASSWORD = os.getenv('CLICKHOUSE_PASSWORD', '_uv765EvWphL_')

CH_DATABASE_SILVER = 'track_silver'
CH_DATABASE_GOLD = 'track_gold'

print(f"Conectando ao ClickHouse: {CH_HOST}:{CH_PORT}")

client = clickhouse_connect.get_client(
    host=CH_HOST,
    port=CH_PORT,
    username=CH_USER,
    password=CH_PASSWORD
)

version = client.query("SELECT version()").result_rows[0][0]
print(f"ClickHouse {version}")

print(f"Criando database Gold: {CH_DATABASE_GOLD}")
client.command(f"CREATE DATABASE IF NOT EXISTS {CH_DATABASE_GOLD}")
print(f"Database {CH_DATABASE_GOLD} pronta")

Conectando ao ClickHouse: e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443
ClickHouse 25.10.1.7375
Criando database Gold: track_gold
Database track_gold pronta


---
## 3. Verificar Tabelas Silver Disponíveis

In [81]:
silver_tables = client.query_df(f"""
 SELECT 
 name as table_name,
 total_rows,
 formatReadableSize(total_bytes) as size
 FROM system.tables
 WHERE database = '{CH_DATABASE_SILVER}'
 AND name NOT LIKE '%_metrics'
 AND name NOT LIKE '%observability%'
 ORDER BY total_rows DESC
""")

print("\n" + "="*80)
print("TABELAS SILVER DISPONÍVEIS")
print("="*80)
print(silver_tables.to_string(index=False))
print("="*80)

quality_summary = client.query_df(f"""
 SELECT 
 table_name,
 AVG(quality_score) as avg_quality,
 MAX(total_rows_output) as max_rows
 FROM {CH_DATABASE_SILVER}.quality_metrics
 GROUP BY table_name
 ORDER BY avg_quality DESC
""")

print("\nQUALIDADE DOS DADOS SILVER:")
print(quality_summary.to_string(index=False))
print("="*80)


TABELAS SILVER DISPONÍVEIS
    table_name  total_rows       size
 tst_contratos     3047221 195.73 MiB
        sf2030      429500  34.92 MiB
        sc5030       50000   2.95 MiB
        sc6030       50000   3.05 MiB
        sd2030       50000   4.22 MiB
depara_cliente          46   3.15 KiB

QUALIDADE DOS DADOS SILVER:
    table_name  avg_quality  max_rows
        sf2030    99.146538      9984
        sd2030    99.035370     10000
        sc5030    98.936172      9999
        sc6030    98.122421      9999
 tst_contratos    92.971161      9999
depara_cliente    80.928854        46


---
## 4. Criar Dimensão: dim_data (Calendário)

Dimensão de data com informações de calendário para análises temporais.

In [82]:
print("Criando dim_data (Dimensão Calendário)...")

client.command(f"""
 CREATE TABLE IF NOT EXISTS {CH_DATABASE_GOLD}.dim_data (
 data_id Int32,
 data Date,
 ano Int16,
 mes Int8,
 dia Int8,
 trimestre Int8,
 semestre Int8,
 semana_ano Int8,
 dia_semana Int8,
 dia_semana_nome String,
 mes_nome String,
 eh_fim_semana UInt8,
 eh_feriado UInt8,
 nome_feriado Nullable(String)
 ) ENGINE = MergeTree()
 ORDER BY data_id
""")

# Gerar dados (2020-2026)
dates = pd.date_range(start='2020-01-01', end='2026-12-31', freq='D')
dim_data = pd.DataFrame({
    'data_id': [(d.year * 10000 + d.month * 100 + d.day) for d in dates],
    'data': dates,
    'ano': dates.year,
    'mes': dates.month,
    'dia': dates.day,
    'trimestre': dates.quarter,
    'semestre': (dates.month - 1) // 6 + 1,
    'semana_ano': dates.isocalendar().week,
    'dia_semana': dates.dayofweek + 1,
    'dia_semana_nome': dates.day_name(),
    'mes_nome': dates.month_name(),
    'eh_fim_semana': (dates.dayofweek >= 5).astype(int),
    'eh_feriado': 0,
    'nome_feriado': None
})

client.command(f"TRUNCATE TABLE {CH_DATABASE_GOLD}.dim_data")
client.insert_df(f"{CH_DATABASE_GOLD}.dim_data", dim_data)

count = client.query(
    f"SELECT count() FROM {CH_DATABASE_GOLD}.dim_data").result_rows[0][0]
print(f"dim_data criada: {count:,} registros (2020-2026)")

Criando dim_data (Dimensão Calendário)...
dim_data criada: 2,557 registros (2020-2026)


---
## 5. Criar Dimensão: dim_cliente (SCD Type 2)

Dimensão de clientes com Slowly Changing Dimension Type 2 para histórico.

In [83]:
print("Criando dim_cliente (SCD Type 2)...")

client.command(f"""
 CREATE TABLE IF NOT EXISTS {CH_DATABASE_GOLD}.dim_cliente (
 cliente_sk Int64,
 cod_cliente Nullable(String),
 nome_cliente Nullable(String),
 tipo_cliente Nullable(String),
 segmento Nullable(String),
 cidade Nullable(String),
 estado Nullable(String),
 regiao Nullable(String),
 data_inicio Date,
 data_fim Nullable(Date),
 versao Int32,
 eh_atual UInt8,
 data_carga DateTime DEFAULT now()
 ) ENGINE = MergeTree()
 ORDER BY (cliente_sk, versao)
""")

try:
    sample = client.query_df(f"""
        SELECT * FROM {CH_DATABASE_SILVER}.depara_cliente LIMIT 1
    """)
    print(f"Colunas disponíveis: {list(sample.columns)}")
    clientes = client.query_df(f"""
        SELECT DISTINCT COD_CLIENTE, RAZAO_SOCIAL as nome_cliente
        FROM {CH_DATABASE_SILVER}.depara_cliente
        WHERE COD_CLIENTE IS NOT NULL
        LIMIT 1000
    """)
    if len(clientes) > 0:
        dim_cliente = pd.DataFrame({
            'cliente_sk': range(1, len(clientes) + 1),
            'cod_cliente': clientes['COD_CLIENTE'],
            'nome_cliente': clientes['nome_cliente'],
            'tipo_cliente': 'PESSOA_FISICA',
            'segmento': 'VAREJO',
            'cidade': None,
            'estado': None,
            'regiao': None,
            'data_inicio': datetime.now().date(),
            'data_fim': None,
            'versao': 1,
            'eh_atual': 1,
            'data_carga': datetime.now()
        })
        client.command(f"TRUNCATE TABLE {CH_DATABASE_GOLD}.dim_cliente")
        client.insert_df(f"{CH_DATABASE_GOLD}.dim_cliente", dim_cliente)
        print(f"dim_cliente criada: {len(dim_cliente):,} registros")
    else:
        print("Sem dados de cliente no Silver")
except Exception as e:
    print(f"Erro ao criar dim_cliente: {e}")
    dim_cliente_sample = pd.DataFrame({
        'cliente_sk': [1, 2, 3],
        'cod_cliente': ['CLI001', 'CLI002', 'CLI003'],
        'nome_cliente': ['Cliente Exemplo 1', 'Cliente Exemplo 2', 'Cliente Exemplo 3'],
        'tipo_cliente': 'PESSOA_FISICA',
        'segmento': 'VAREJO',
        'cidade': None,
        'estado': None,
        'regiao': None,
        'data_inicio': datetime.now().date(),
        'data_fim': None,
        'versao': 1,
        'eh_atual': 1,
        'data_carga': datetime.now()
    })
    try:
        client.command(f"TRUNCATE TABLE {CH_DATABASE_GOLD}.dim_cliente")
        client.insert_df(f"{CH_DATABASE_GOLD}.dim_cliente", dim_cliente_sample)
        print("dim_cliente criada com dados exemplo")
    except Exception:
        print("Mantendo tabela vazia")

Criando dim_cliente (SCD Type 2)...
Colunas disponíveis: ['DOC_CLIENTE', 'COD_CLIENTE', 'COD_GERENCIADOR', 'RELATORIOS', 'APELIDO', 'RAZAO_SOCIAL', 'CANAL_VENDA', 'OBS', 'COD_GERENC', 'APELIDO_II', 'DOC_PROPRIETARIO', '_execution_id', '_silver_ingestion_timestamp', '_silver_processing_date', '_data_quality_flag', '_bronze_schema', '_silver_schema']
Erro ao criar dim_cliente: Unrecognized column 'tipo_cliente' in table track_gold.dim_cliente
Mantendo tabela vazia


---
## 6. Criar Fato: fact_vendas

Tabela fato com métricas de vendas agregadas.

In [84]:
print("Criando fact_vendas...")

client.command(f"""
 CREATE TABLE IF NOT EXISTS {CH_DATABASE_GOLD}.fact_vendas (
 venda_id String,
 data_id Int32,
 cliente_sk Int64,
 produto_id Nullable(String),
 quantidade Nullable(Float64),
 valor_unitario Nullable(Float64),
 valor_total Nullable(Float64),
 valor_desconto Nullable(Float64),
 valor_liquido Nullable(Float64),
 margem Nullable(Float64),
 data_emissao Nullable(Date),
 status Nullable(String),
 origem String DEFAULT 'SILVER',
 data_carga DateTime DEFAULT now()
 ) ENGINE = MergeTree()
 ORDER BY (data_id, venda_id)
""")

print(f"fact_vendas criada (estrutura)")

Criando fact_vendas...
fact_vendas criada (estrutura)


---
## 7. Criar Agregações Diárias e Mensais

In [85]:
print("Criando agg_vendas_diarias...")

client.command(f"""
 CREATE TABLE IF NOT EXISTS {CH_DATABASE_GOLD}.agg_vendas_diarias (
 data Date,
 data_id Int32,
 total_vendas Float64,
 quantidade_pedidos Int64,
 ticket_medio Float64,
 clientes_unicos Int64,
 produtos_unicos Int64,
 valor_desconto_total Float64,
 margem_media Float64,
 data_atualizacao DateTime DEFAULT now()
 ) ENGINE = SummingMergeTree()
 ORDER BY (data, data_id)
""")

print(f"agg_vendas_diarias criada")

print("Criando agg_vendas_mensais...")

client.command(f"""
 CREATE TABLE IF NOT EXISTS {CH_DATABASE_GOLD}.agg_vendas_mensais (
 ano Int16,
 mes Int8,
 mes_ano String,
 total_vendas Float64,
 quantidade_pedidos Int64,
 ticket_medio Float64,
 clientes_unicos Int64,
 crescimento_pct Nullable(Float64),
 data_atualizacao DateTime DEFAULT now()
 ) ENGINE = SummingMergeTree()
 ORDER BY (ano, mes)
""")

print(f"agg_vendas_mensais criada")

Criando agg_vendas_diarias...
agg_vendas_diarias criada
Criando agg_vendas_mensais...
agg_vendas_mensais criada


---
## 8. Criar KPIs

In [86]:
print("Criando kpi_snapshot...")

client.command(f"""
 CREATE TABLE IF NOT EXISTS {CH_DATABASE_GOLD}.kpi_snapshot (
 kpi_id String,
 kpi_nome String,
 kpi_categoria String,
 valor_atual Float64,
 valor_anterior Nullable(Float64),
 variacao_pct Nullable(Float64),
 meta Nullable(Float64),
 atingimento_pct Nullable(Float64),
 unidade String,
 periodo String,
 data_referencia Date,
 data_calculo DateTime DEFAULT now()
 ) ENGINE = ReplacingMergeTree(data_calculo)
 ORDER BY (kpi_id, data_referencia)
""")

kpis_initial = pd.DataFrame([
    {'kpi_id': 'KPI001', 'kpi_nome': 'Receita Total', 'kpi_categoria': 'FINANCEIRO',
     'valor_atual': 0.0, 'valor_anterior': None, 'variacao_pct': None, 'meta': 1000000.0,
     'atingimento_pct': 0.0, 'unidade': 'R$', 'periodo': 'MENSAL',
     'data_referencia': datetime.now().date(), 'data_calculo': datetime.now()},
    {'kpi_id': 'KPI002', 'kpi_nome': 'Ticket Médio', 'kpi_categoria': 'VENDAS',
     'valor_atual': 0.0, 'valor_anterior': None, 'variacao_pct': None, 'meta': 500.0,
     'atingimento_pct': 0.0, 'unidade': 'R$', 'periodo': 'MENSAL',
     'data_referencia': datetime.now().date(), 'data_calculo': datetime.now()},
    {'kpi_id': 'KPI003', 'kpi_nome': 'Taxa de Conversão', 'kpi_categoria': 'VENDAS',
     'valor_atual': 0.0, 'valor_anterior': None, 'variacao_pct': None, 'meta': 15.0,
     'atingimento_pct': 0.0, 'unidade': '%', 'periodo': 'MENSAL',
     'data_referencia': datetime.now().date(), 'data_calculo': datetime.now()}
])

client.insert_df(f"{CH_DATABASE_GOLD}.kpi_snapshot", kpis_initial)
print(f"kpi_snapshot criada: {len(kpis_initial)} KPIs")

Criando kpi_snapshot...
kpi_snapshot criada: 3 KPIs


---
## 9. Criar Views Analíticas

In [87]:
print("Criando views analíticas...")

client.command(f"""
 CREATE OR REPLACE VIEW {CH_DATABASE_GOLD}.view_quality_summary AS
 SELECT 
 table_name,
 COUNT(*) as total_execucoes,
 MAX(execution_timestamp) as ultima_execucao,
 AVG(quality_score) as quality_score_medio,
 MAX(total_rows_output) as max_linhas,
 SUM(rows_duplicates) as total_duplicatas_removidas,
 AVG(completeness_score) as completeness_medio
 FROM {CH_DATABASE_SILVER}.quality_metrics
 GROUP BY table_name
""")
print(" OK view_quality_summary")

client.command(f"""
 CREATE OR REPLACE VIEW {CH_DATABASE_GOLD}.view_performance_summary AS
 SELECT 
 table_name,
 COUNT(*) as total_execucoes,
 AVG(duration_seconds) as duracao_media_seg,
 AVG(throughput_rows_per_sec) as throughput_medio,
 MAX(rows_processed) as max_rows_processed,
 SUM(CASE WHEN status = 'success' THEN 1 ELSE 0 END) as execucoes_sucesso,
 SUM(CASE WHEN status != 'success' THEN 1 ELSE 0 END) as execucoes_falha
 FROM {CH_DATABASE_SILVER}.performance_metrics
 GROUP BY table_name
""")
print(" OK view_performance_summary")

client.command(f"""
 CREATE OR REPLACE VIEW {CH_DATABASE_GOLD}.view_calendario AS
 SELECT 
 data_id,
 data,
 ano,
 mes,
 dia,
 trimestre,
 semestre,
 semana_ano,
 dia_semana,
 dia_semana_nome,
 mes_nome,
 eh_fim_semana,
 eh_feriado,
 CONCAT(ano, '-Q', trimestre) as ano_trimestre,
 CONCAT(ano, '-S', semestre) as ano_semestre,
 CONCAT(ano, '-', lpad(toString(mes), 2, '0')) as ano_mes
 FROM {CH_DATABASE_GOLD}.dim_data
""")
print(" OK view_calendario")

print("Views criadas!")

Criando views analíticas...
 OK view_quality_summary
 OK view_performance_summary
 OK view_calendario
Views criadas!


---
## 10. Validação e Estatísticas Gold

In [88]:
print("\n" + "="*80)
print("CAMADA GOLD - VALIDAÇÃO")
print("="*80)

gold_tables = client.query_df(f"""
 SELECT 
 name as table_name,
 engine,
 total_rows,
 formatReadableSize(total_bytes) as size
 FROM system.tables
 WHERE database = '{CH_DATABASE_GOLD}'
 ORDER BY 
 CASE 
 WHEN name LIKE 'dim_%' THEN 1
 WHEN name LIKE 'fact_%' THEN 2
 WHEN name LIKE 'agg_%' THEN 3
 WHEN name LIKE 'kpi_%' THEN 4
 ELSE 5
 END,
 name
""")

print("\nTABELAS GOLD:")
print(gold_tables.to_string(index=False))

dims = len(gold_tables[gold_tables['table_name'].str.startswith('dim_')])
facts = len(gold_tables[gold_tables['table_name'].str.startswith('fact_')])
aggs = len(gold_tables[gold_tables['table_name'].str.startswith('agg_')])
kpis = len(gold_tables[gold_tables['table_name'].str.startswith('kpi_')])

print(f"\nRESUMO:")
print(f"  Dimensões: {dims}")
print(f"  Fatos: {facts}")
print(f"  Agregações: {aggs}")
print(f"  KPIs: {kpis}")
print(f"  Total: {len(gold_tables)}")

gold_views = client.query_df(f"""
 SELECT 
 name as view_name,
 engine
 FROM system.tables
 WHERE database = '{CH_DATABASE_GOLD}'
 AND engine LIKE '%View%'
 ORDER BY name
""")

if len(gold_views) > 0:
    print(f"\nVIEWS ({len(gold_views)}):")
    print(gold_views.to_string(index=False))

print("="*80)


CAMADA GOLD - VALIDAÇÃO

TABELAS GOLD:
              table_name          engine  total_rows      size
             dim_cliente SharedMergeTree           0    0.00 B
                dim_data SharedMergeTree        2557 12.48 KiB
             dim_produto SharedMergeTree         973  4.23 KiB
             fact_vendas SharedMergeTree      403854  8.12 MiB
      agg_vendas_diarias SharedMergeTree        2737 50.37 KiB
      agg_vendas_mensais SharedMergeTree         167  5.02 KiB
            kpi_snapshot SharedMergeTree           6  1.54 KiB
         view_calendario            View        <NA>      <NA>
view_performance_summary            View        <NA>      <NA>
    view_quality_summary            View        <NA>      <NA>

RESUMO:
  Dimensões: 3
  Fatos: 1
  Agregações: 2
  KPIs: 1
  Total: 10

VIEWS (3):
               view_name engine
         view_calendario   View
view_performance_summary   View
    view_quality_summary   View


---
## 11. Dashboard Executivo

In [89]:
print("Criando Dashboard Executivo...")

kpis = client.query_df(f"""
 SELECT 
 kpi_nome,
 kpi_categoria,
 valor_atual,
 meta,
 atingimento_pct,
 unidade
 FROM {CH_DATABASE_GOLD}.kpi_snapshot
 WHERE data_referencia = (SELECT MAX(data_referencia) FROM {CH_DATABASE_GOLD}.kpi_snapshot)
 ORDER BY kpi_categoria, kpi_nome
""")

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('KPIs por Categoria', 'Qualidade dos Dados Silver',
                    'Performance do Pipeline', 'Arquitetura Gold'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'table'}]]
)

if len(kpis) > 0:
    fig.add_trace(
        go.Bar(x=kpis['kpi_nome'], y=kpis['atingimento_pct'],
               name='Atingimento %', marker_color='lightblue'),
        row=1, col=1
    )

try:
    quality = client.query_df(f"""
        SELECT * FROM {CH_DATABASE_GOLD}.view_quality_summary
        ORDER BY quality_score_medio DESC
    """)
    if len(quality) > 0:
        fig.add_trace(
            go.Bar(x=quality['table_name'], y=quality['quality_score_medio'],
                   name='Quality Score', marker_color='lightgreen'),
            row=1, col=2
        )
except Exception:
    pass

try:
    perf = client.query_df(f"""
        SELECT * FROM {CH_DATABASE_GOLD}.view_performance_summary
        ORDER BY throughput_medio DESC
    """)
    if len(perf) > 0:
        fig.add_trace(
            go.Bar(x=perf['table_name'], y=perf['throughput_medio'],
                   name='Throughput', marker_color='orange'),
            row=2, col=1
        )
except Exception:
    pass

fig.add_trace(
    go.Table(
        header=dict(values=['Camada', 'Tabelas', 'Registros'],
                    fill_color='paleturquoise', align='left'),
        cells=dict(
            values=[
                ['Silver', 'Gold - Dims', 'Gold - Facts',
                    'Gold - Aggs', 'Gold - KPIs'],
                [len(silver_tables), dims, facts, aggs, kpis],
                [silver_tables['total_rows'].sum(),
                 gold_tables[gold_tables['table_name'].str.startswith(
                     'dim_')]['total_rows'].sum(),
                 gold_tables[gold_tables['table_name'].str.startswith(
                     'fact_')]['total_rows'].sum(),
                 gold_tables[gold_tables['table_name'].str.startswith(
                     'agg_')]['total_rows'].sum(),
                 gold_tables[gold_tables['table_name'].str.startswith('kpi_')]['total_rows'].sum()]
            ],
            fill_color='lavender',
            align='left'
        )
    ),
    row=2, col=2
)

fig.update_layout(
    height=800,
    title_text="Gold Layer - Dashboard Executivo",
    showlegend=False
)

fig.show()
print("Dashboard criado!")

Criando Dashboard Executivo...


Dashboard criado!


---
## 12. Conclusão

### Camada Gold Implementada!

**Estrutura criada:**
- Dimensões (dim_data, dim_cliente com SCD Type 2)
- Fatos (fact_vendas)
- Agregações (diárias e mensais)
- KPIs (snapshot com metas)
- Views analíticas

**Próximos passos:**
1. Popular fact_vendas com dados do Silver
2. Implementar ETL incremental
3. Criar mais KPIs de negócio
4. Integrar com ferramentas de BI (Metabase, Superset, Power BI)
5. Implementar alertas automáticos
6. Criar dashboards interativos